In [ ]:
import json
import numpy as np
import ollama
import chromadb
from pathlib import Path

cleaned_documents_file = Path("../data/processed/cleaned_documents.json")
chunked_documents_file = Path("../data/processed/chunked_documents.json")
embedding_file = Path("../data/processed/embeddings.npy")
chroma_path = Path("../data/chroma")

print("Setup complete")

In [ ]:
#loading documents

with open(cleaned_documents_file, "r", encoding="utf-8") as f:
    cleaned_documents = json.load(f)

print("Total documents:", len(cleaned_documents))

In [ ]:
#loading chunks 

with open(chunked_documents_file, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

print("Total chunks:", len(all_chunks))

In [ ]:

#loading embeddings 

embeddings_array = np.load(embedding_file)

print("Embeddings shape:", embeddings_array.shape)

In [ ]:
#checking if chunks and embeddings are loaded and matches
print("Chunks:", len(all_chunks))
print("Embeddings:", len(embeddings_array))

assert len(all_chunks) == len(embeddings_array)

print("Chunk and embedding counts match")

In [ ]:
#creating a chromadb client to store the embeddings

client = chromadb.PersistentClient(
    path=str(chroma_path)
)

print("ChromaDB client created successfully")

In [ ]:
#creating a collection to store into chromadb

collection = client.get_or_create_collection(
    name="documents"
)

print("Collection:", collection.name)
print("Current count:", collection.count())

In [ ]:
#storing the chunks in batched of size = 100

batch_size = 100

for start in range(0, len(all_chunks), batch_size):

    end = min(start + batch_size, len(all_chunks))

    batch_chunks = all_chunks[start:end]
    batch_embeddings = embeddings_array[start:end]

    collection.add(
        ids=[chunk["chunk_id"] for chunk in batch_chunks],
        embeddings=batch_embeddings.tolist(),
        documents=[chunk["text"] for chunk in batch_chunks],
        metadatas=[
            {
                "doc_id": chunk["doc_id"],
                "source": chunk["source"],
                "chunk_index": chunk["chunk_index"]
            }
            for chunk in batch_chunks
        ]
    )

    print(f"Inserted {end}/{len(all_chunks)} chunks")

In [ ]:
#checking the collection count

print("Chroma collection count:", collection.count())

In [ ]:
#Fetching the first chunk for verifying 

result = collection.get(
    ids=[all_chunks[0]["chunk_id"]],
    include=["embeddings", "documents", "metadatas"]
)

print("ID:", result["ids"][0])
print("Embedding dimensions:", len(result["embeddings"][0]))
print("Document:", result["documents"][0][:300])
print("Metadata:", result["metadatas"][0])

In [ ]:
##Retrieval 


#Testing whether the database can retrieve the relevant chunks for a user query

In [ ]:
#setting a sample query
query = "PSMA targeted radiogland therapy in prostate cancer "

print(query)

In [ ]:
context = "\n\n".join(results["documents"][0])

print(context) 

In [ ]:
question = "PSMA targeted radioligand therapy in prostate cancer"

In [ ]:
prompt = f"""
You are a biomedical research assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

In [ ]:
#getting an response from the local ollama model 

response = ollama.chat(
    model = "qwen2.5:3b",
    messages= [
        {

            "role": "user",
            "content": prompt
            
        }
    ]
)

answer = response["message"]["content"]

print(answer)

In [ ]:
#calculating the time taken by the model to give the response

import time 

start = time.time()
response = ollama.chat(
    model = "qwen2.5:3b",
    messages=[
        {
            "role":"user",
            "content": prompt
        }
    ]
)

end = time.time()
print("Generation time:", end - start, "seconds")
print(response["message"]["content"])



In [ ]:
import time

# 1. Query embedding time 
start = time.time()

query_embedding = ollama.embed(
    model="nomic-embed-text",
    input=question
)["embeddings"][0]

embedding_time = time.time() - start


# 2. Vector retrieval time
start = time.time()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

retrieval_time = time.time() - start


# 3. Context assembly time
start = time.time()

context = "\n\n".join(results["documents"][0])

context_time = time.time() - start


# 4. Prompt construction time
start = time.time()

prompt = f"""
You are a biomedical research assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

prompt_time = time.time() - start


print("Query embedding:", embedding_time, "seconds")
print("Retrieval:", retrieval_time, "seconds")
print("Context assembly:", context_time, "seconds")
print("Prompt construction:", prompt_time, "seconds")

In [ ]:
print("Prompt tokens:", response.get("prompt_eval_count"))
print("Generated tokens:", response.get("eval_count"))

print("Prompt evaluation time:",
      response.get("prompt_eval_duration", 0) / 1e9,
      "seconds")

print("Generation evaluation time:",
      response.get("eval_duration", 0) / 1e9,
      "seconds")

print("Total duration:",
      response.get("total_duration", 0) / 1e9,
      "seconds")

In [ ]:
#getting an response from the external LLM gemini 3.6
import ollama 
from dotenv import load_dotenv
import os 
from google import genai


load_dotenv()
 
api_key = os.getenv("GEMINI_API_KEY")
print("api key loaded:", api_key is not None)


client = genai.Client(api_key = api_key)
print("Gemini client created")

In [ ]:
question = "PSMA targeted radioligand therapy in prostate cancer"

query_embedding = ollama.embed(
    model="nomic-embed-text",
    input=question
)["embeddings"][0]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

context = "\n\n".join(results["documents"][0])

prompt = f"""
You are a biomedical research assistant.

Answer the question using only the provided context.

Context:
{context}

Question:
{question}

Answer:
"""

print("Retrieved chunks:", len(results["documents"][0]))
print("Prompt ready.")

In [ ]:
#checking the time taking by the gemini 3.6 model to provide an reponse

import time 

start = time.time()

response = client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = prompt
)

end = time.time()

print("Gemini generation time:" , end - start , "seconds")
print()
print(response.text)   

In [ ]:
## RAG EVALUATION 

In [ ]:
evaluation_question = "What is PSMA-targeted radioligand therapy used for in prostate cancer?"

print(evaluation_question)

In [ ]:
query_embedding = ollama.embed(
    model="nomic-embed-text",
    input=evaluation_question
)["embeddings"][0]

eval_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

eval_context = "\n\n".join(eval_results["documents"][0])

print(eval_context) 

In [ ]:
evaluation_results = []

for item in evaluation_questions:

    question =  item["question"]

    query_embedding = ollama.embed(
        model = "nomic-embed-text",
        input = question

    )["embeddings"][0]


    results =  collection.query(

        query_embeddings = [query_embedding], 
        n_results = 5
    
    )


    evaluation_results.append({
        "id" : item["id"],
        "question": question,
        "category": item["category"],
        "retrieved_ids": results["ids"][0],
        "retrieved_documents": results["documents"][0],
        "distances": results["distances"][0]
    })

    print("Evaluated questions:" , len(evaluation_results))


    





In [ ]:
first_result = evaluation_results[0]

print("Question:", first_result["question"])
print()
print("Retrieved IDs:")
for i, doc_id in enumerate(first_result["retrieved_ids"], start=1):
    print(i, doc_id)

print()
print("Distances:")
for i, distance in enumerate(first_result["distances"], start=1):
    print(i, distance)